# Análise Exploratória de Dados (EDA) - Small Dataset
Vamos analisar os dados fornecidos (data_small) para o distrito de Braga.
O objetivo é carregar os CSVs, validar a integridade das janelas temporais, medir pesos face às capacidades e mapear geograficamente as paragens.

In [13]:
%pip install pandas folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import folium

# Carregar os datasets a partir da pasta 'data_small'
pasta = "../data_small/"
df_localities = pd.read_csv(pasta + "localities.csv")
df_locations = pd.read_csv(pasta + "locations.csv")
df_producers = pd.read_csv(pasta + "producers.csv")
df_products = pd.read_csv(pasta + "products.csv")
df_orders = pd.read_csv(pasta + "orders.csv")
df_order_items = pd.read_csv(pasta + "order_items.csv")
df_vehicles = pd.read_csv(pasta + "vehicles.csv")

print("Todos os ficheiros CSV foram carregados com sucesso da pasta data_small!")

Todos os ficheiros CSV foram carregados com sucesso da pasta data_small!


## 1. Validação de Janelas Temporais
Vamos garantir que a disponibilidade do produto não termina antes de começar, e que a entrega é feita após o produto estar disponível para recolha.

In [15]:
colunas_data = ['availability_start', 'availability_end', 'delivery_start', 'delivery_end']
for col in colunas_data:
    df_order_items[col] = pd.to_datetime(df_order_items[col])

valid_avail = (df_order_items['availability_end'] >= df_order_items['availability_start']).all()
valid_deliv_win = (df_order_items['delivery_end'] >= df_order_items['delivery_start']).all()
valid_deliv = (df_order_items['delivery_end'] >= df_order_items['availability_start']).all()
n_start_antes = (df_order_items['delivery_start'] < df_order_items['availability_start']).sum()

print(f"Janelas de disponibilidade válidas? {valid_avail}")
print(f"Janelas de entrega válidas? {valid_deliv_win}")
print(f"Entregas terminam após o início da disponibilidade? {valid_deliv}")
print(f"Itens com delivery_start antes de availability_start: {n_start_antes}")

Janelas de disponibilidade válidas? True
Janelas de entrega válidas? True
Entregas terminam após o início da disponibilidade? True
Itens com delivery_start antes de availability_start: 0


## 2. Pesos vs Capacidades dos Veículos
Temos de garantir que as encomendas cabem nos veículos.

In [16]:
df_pesos = pd.merge(df_order_items, df_products, on='product_id')
df_pesos['peso_total'] = df_pesos['quantity'] * df_pesos['unit_weight_kg']

peso_maximo_item = df_pesos['peso_total'].max()
capacidade_minima_veiculo = df_vehicles['capacity_kg'].min()
peso_total = df_pesos['peso_total'].sum()
capacidade_frota = df_vehicles['capacity_kg'].sum()

print(f"Item mais pesado: {peso_maximo_item:.2f} kg")
print(f"Veículo com menor capacidade suporta: {capacidade_minima_veiculo:.2f} kg")
print(f"Peso total dos pedidos: {peso_total:.1f} kg | Capacidade total da frota: {capacidade_frota:.1f} kg "
      f"({100*peso_total/capacidade_frota:.1f}% de ocupação mínima necessária)")

if peso_maximo_item <= capacidade_minima_veiculo:
    print("Todas as encomendas individuais cabem em qualquer veículo da frota.")
else:
    print("AVISO: Existem itens que ultrapassam a capacidade de algum veículo!")

Item mais pesado: 21.80 kg
Veículo com menor capacidade suporta: 250.00 kg
Peso total dos pedidos: 580.0 kg | Capacidade total da frota: 550.0 kg (105.5% de ocupação mínima necessária)
Todas as encomendas individuais cabem em qualquer veículo da frota.


## 3. Mapeamento das Paragens
Visualização dos produtores, clientes e centros de recolha no distrito de Braga.

In [17]:
mapa = folium.Map(location=[41.55, -8.4], zoom_start=10)

#Cores que os ponto irão receber no mapa
cores = {
    'producer': 'green', 
    'collection_center': 'red', 
    'delivery_location': 'blue', 
    'customer': 'blue', 
    'delivery': 'blue', 
    'center': 'red'
}

for index, row in df_locations.iterrows():
    tipo = row['location_type']
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"{row['name']} ({tipo})",
        icon=folium.Icon(color=cores.get(tipo, 'gray'), icon='info-sign')
    ).add_to(mapa)

In [18]:
# Mostra o mapa
mapa